In [1]:
from run.ViolenceDetection import ViolenceDetector
import os


In [2]:
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score, 
    precision_score, recall_score, f1_score, roc_curve, auc, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def compute_metrics(y_true, y_pred, y_probs=None, threshold=0.5):
    """
    Compute comprehensive classification metrics
    
    Args:
        y_true: Ground truth labels (0/1)
        y_pred: Predicted labels (0/1)
        y_probs: Predicted probabilities (optional, for ROC/AUC)
        threshold: Classification threshold
    """
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Basic Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Additional Metrics
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # False Negative Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # False Positive Rate
    
    # ROC-AUC (if probabilities provided)
    roc_auc = None
    if y_probs is not None:
        roc_auc = roc_auc_score(y_true, y_probs)
    
    # Print Results
    print(f"\n{'='*50}")
    print(f"CONFUSION MATRIX (Threshold: {threshold})")
    print(f"{'='*50}")
    print(f"TP: {tp:4d} | FP: {fp:4d}")
    print(f"FN: {fn:4d} | TN: {tn:4d}")
    print(f"\n{'='*50}")
    print(f"METRICS")
    print(f"{'='*50}")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"F1 Score:    {f1:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"FPR:         {fpr:.4f}")
    print(f"FNR:         {fnr:.4f}")
    if roc_auc is not None:
        print(f"ROC-AUC:     {roc_auc:.4f}")
    
    # Classification Report
    print(f"\n{'='*50}")
    print(f"CLASSIFICATION REPORT")
    print(f"{'='*50}")
    print(classification_report(y_true, y_pred, target_names=['No Violence', 'Violence']))
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'specificity': specificity,
        'fpr': fpr,
        'fnr': fnr,
        'roc_auc': roc_auc,
        'cm': cm
    }

def plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Violence', 'Violence'],
                yticklabels=['No Violence', 'Violence'])
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    logger.info(f"Confusion matrix saved to {save_path}")
    plt.close()

def plot_roc_curve(y_true, y_probs, save_path='roc_curve.png'):
    """Plot and save ROC curve"""
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    logger.info(f"ROC curve saved to {save_path}")
    plt.close()

def threshold_analysis(y_true, y_probs, save_path='threshold_analysis.png'):
    """Analyze metrics across different thresholds"""
    thresholds = np.arange(0, 1.01, 0.05)
    metrics_by_threshold = {
        'precision': [],
        'recall': [],
        'f1': [],
        'accuracy': []
    }
    
    for thresh in thresholds:
        y_pred = (y_probs >= thresh).astype(int)
        metrics_by_threshold['precision'].append(precision_score(y_true, y_pred, zero_division=0))
        metrics_by_threshold['recall'].append(recall_score(y_true, y_pred, zero_division=0))
        metrics_by_threshold['f1'].append(f1_score(y_true, y_pred, zero_division=0))
        metrics_by_threshold['accuracy'].append(accuracy_score(y_true, y_pred))
    
    plt.figure(figsize=(10, 6))
    for metric, values in metrics_by_threshold.items():
        plt.plot(thresholds, values, marker='o', label=metric)
    plt.xlabel('Classification Threshold')
    plt.ylabel('Metric Value')
    plt.title('Metrics vs Classification Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    logger.info(f"Threshold analysis saved to {save_path}")
    plt.close()

def analyze(y_pred, y_true, y_scores, dataset_name=""):
    
    y_pred = (np.array(y_scores) >= 0.5).astype(int)

    # Compute metrics
    metrics = compute_metrics(y_true, y_pred, y_scores, threshold=0.5)

    # Generate visualizations
    plot_confusion_matrix(y_true, y_pred, save_path=f'{dataset_name}_confusion_matrix.png')
    plot_roc_curve(y_true, y_scores, save_path=f'{dataset_name}_roc_curve.png')
    threshold_analysis(y_true, y_scores, save_path=f'{dataset_name}_threshold_analysis.png')

    return metrics

In [4]:
VAL_DIRECTORY = "RWF_val"  # Update this path to your validation directory
TP, FP, TN, FN = 0, 0, 0, 0

y_true = []
y_scores = []

import time



#Fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/Fight"):
    video_path = os.path.join(VAL_DIRECTORY, "fight", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(1)  # Assuming this is a fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob >= 0.5:  # Threshold for classification
        TP += 1
    else:
        FN += 1
    time.sleep(0.05)  # Small delay to prevent resource exhaustion

#Non-fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/NonFight"):
    video_path = os.path.join(VAL_DIRECTORY, "NonFight", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(0)  # Assuming this is a non-fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob < 0.5:  # Threshold for classification
        TN += 1
    else:
        FP += 1

    time.sleep(0.05)  # Small delay to prevent resource exhaustion

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)


2026-03-12 09:41:13,071 - run.ViolenceDetection - INFO - Video detected: 1280x720 @ 30.0fps
C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
2026-03-12 09:41:16,329 - run.ViolenceDetection - INFO - Validation complete. Total frames: 150, Max violence probability: 0.6942, Climax frame: 148, Processing time: 2.95 seconds
2026-03-12 09:41:16,398 - run.ViolenceDetection - INFO - Video detected: 1280x720 @ 30.0fps
2026-03-12 09:41:19,966 - run.ViolenceDetection - INFO - Validation complete. Total frames: 150, Max violence probability: 0.7684, Climax frame: 67, Processing time: 3.29 seconds
2026-03-12 09:41:20,030 - run.ViolenceDetection - INFO - Video detected: 1280x720 @ 30.0fps
2026-03-12 09:41:23,516 - run.ViolenceDetection - INFO - Validation co


CONFUSION MATRIX (Threshold: 0.5)
TP:  167 | FP:   42
FN:   33 | TN:  158

METRICS
Accuracy:    0.8125
Precision:   0.7990
Recall:      0.8350
F1 Score:    0.8166
Specificity: 0.7900
FPR:         0.2100
FNR:         0.1650
ROC-AUC:     0.8861

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 No Violence       0.83      0.79      0.81       200
    Violence       0.80      0.83      0.82       200

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400



2026-03-12 10:02:16,394 - __main__ - INFO - Confusion matrix saved to RWF_val_confusion_matrix.png
2026-03-12 10:02:16,675 - __main__ - INFO - ROC curve saved to RWF_val_roc_curve.png
2026-03-12 10:02:17,170 - __main__ - INFO - Threshold analysis saved to RWF_val_threshold_analysis.png


In [3]:
VAL_DIRECTORY = "hkfval"
import time

y_true = []
y_scores = []

for video_file in os.listdir(VAL_DIRECTORY):
    if not video_file.endswith(".avi"):
        continue

    fight = video_file[:2] == "fi"
    video_path = os.path.join(VAL_DIRECTORY, video_file)

    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()

    # store results
    y_true.append(1 if fight else 0)
    y_scores.append(max_violence_prob)
    time.sleep(0.05)  # Small delay to prevent resource exhaustion

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)

2026-03-12 09:06:41,932 - run.ViolenceDetection - INFO - Video detected: 360x288 @ 25.0fps
C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
2026-03-12 09:06:43,281 - run.ViolenceDetection - INFO - Validation complete. Total frames: 41, Max violence probability: 0.9410, Climax frame: 37, Processing time: 0.81 seconds
2026-03-12 09:06:43,346 - run.ViolenceDetection - INFO - Video detected: 360x288 @ 25.0fps
2026-03-12 09:06:44,557 - run.ViolenceDetection - INFO - Validation complete. Total frames: 41, Max violence probability: 0.8582, Climax frame: 37, Processing time: 0.93 seconds
2026-03-12 09:06:44,628 - run.ViolenceDetection - INFO - Video detected: 360x288 @ 25.0fps
2026-03-12 09:06:45,748 - run.ViolenceDetection - INFO - Validation complete


CONFUSION MATRIX (Threshold: 0.5)
TP:  478 | FP:  169
FN:   22 | TN:  331

METRICS
Accuracy:    0.8090
Precision:   0.7388
Recall:      0.9560
F1 Score:    0.8335
Specificity: 0.6620
FPR:         0.3380
FNR:         0.0440
ROC-AUC:     0.9247

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 No Violence       0.94      0.66      0.78       500
    Violence       0.74      0.96      0.83       500

    accuracy                           0.81      1000
   macro avg       0.84      0.81      0.80      1000
weighted avg       0.84      0.81      0.80      1000



2026-03-12 09:25:57,478 - __main__ - INFO - Confusion matrix saved to hkfval_confusion_matrix.png
2026-03-12 09:25:57,759 - __main__ - INFO - ROC curve saved to hkfval_roc_curve.png
2026-03-12 09:25:58,345 - __main__ - INFO - Threshold analysis saved to hkfval_threshold_analysis.png


In [4]:
VAL_DIRECTORY = "Peliculas"  # Update this path to your validation directory
TP, FP, TN, FN = 0, 0, 0, 0

y_true = []
y_scores = []

#Fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/fights"):
    video_path = os.path.join(VAL_DIRECTORY, "fights", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(1)  # Assuming this is a fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob >= 0.5:  # Threshold for classification
        TP += 1
    else:
        FN += 1

#Non-fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/nofights"):
    video_path = os.path.join(VAL_DIRECTORY, "nofights", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(0)  # Assuming this is a non-fight video
    y_scores.append(max_violence_prob)
    if max_violence_prob < 0.5:  # Threshold for classification
        TN += 1
    else:
        FP += 1

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)


2026-03-12 09:25:58,393 - run.ViolenceDetection - INFO - Video detected: 720x576 @ 25.0fps
C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
2026-03-12 09:25:59,592 - run.ViolenceDetection - INFO - Validation complete. Total frames: 42, Max violence probability: 0.4872, Climax frame: 37, Processing time: 0.93 seconds
2026-03-12 09:25:59,611 - run.ViolenceDetection - INFO - Video detected: 720x576 @ 25.0fps
2026-03-12 09:26:00,791 - run.ViolenceDetection - INFO - Validation complete. Total frames: 42, Max violence probability: 0.6404, Climax frame: 37, Processing time: 0.91 seconds
2026-03-12 09:26:00,809 - run.ViolenceDetection - INFO - Video detected: 720x480 @ 29.97002997002997fps
2026-03-12 09:26:02,264 - run.ViolenceDetection - INFO - Valida


CONFUSION MATRIX (Threshold: 0.5)
TP:   63 | FP:   14
FN:   37 | TN:   87

METRICS
Accuracy:    0.7463
Precision:   0.8182
Recall:      0.6300
F1 Score:    0.7119
Specificity: 0.8614
FPR:         0.1386
FNR:         0.3700
ROC-AUC:     0.8574

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 No Violence       0.70      0.86      0.77       101
    Violence       0.82      0.63      0.71       100

    accuracy                           0.75       201
   macro avg       0.76      0.75      0.74       201
weighted avg       0.76      0.75      0.74       201



2026-03-12 09:30:06,657 - __main__ - INFO - Confusion matrix saved to Peliculas_confusion_matrix.png
2026-03-12 09:30:07,044 - __main__ - INFO - ROC curve saved to Peliculas_roc_curve.png
2026-03-12 09:30:07,560 - __main__ - INFO - Threshold analysis saved to Peliculas_threshold_analysis.png


In [3]:
VAL_DIRECTORY = "RLVS"  # Update this path to your validation directory

y_true = []
y_scores = []

#Fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/Violence"):
    video_path = os.path.join(VAL_DIRECTORY, "Violence", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(1)  # Assuming this is a fight video
    y_scores.append(max_violence_prob)

#Non-fight validation
for video_file in os.listdir(VAL_DIRECTORY + "/NonViolence"):
    video_path = os.path.join(VAL_DIRECTORY, "NonViolence", video_file)
    detector = ViolenceDetector(video_path)
    max_violence_prob, _ = detector.val()
    y_true.append(0)  # Assuming this is a non-fight video
    y_scores.append(max_violence_prob)

metrics = analyze(y_true, y_true, y_scores, VAL_DIRECTORY)


2026-03-11 08:50:24,495 - run.ViolenceDetection - INFO - Video detected: 1920x1080 @ 15.416153255350068fps
C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
2026-03-11 08:50:27,651 - run.ViolenceDetection - INFO - Validation complete. Total frames: 103, Max violence probability: 0.8500, Climax frame: 78, Processing time: 2.46 seconds
2026-03-11 08:50:27,679 - run.ViolenceDetection - INFO - Video detected: 1920x1080 @ 18.222585507420266fps
2026-03-11 08:50:30,108 - run.ViolenceDetection - INFO - Validation complete. Total frames: 96, Max violence probability: 0.4867, Climax frame: 77, Processing time: 2.08 seconds
2026-03-11 08:50:30,139 - run.ViolenceDetection - INFO - Video detected: 640x360 @ 29.97002997002997fps
2026-03-11 08:50:33,093 - run.


CONFUSION MATRIX (Threshold: 0.5)
TP:  926 | FP:  395
FN:   74 | TN:  605

METRICS
Accuracy:    0.7655
Precision:   0.7010
Recall:      0.9260
F1 Score:    0.7979
Specificity: 0.6050
FPR:         0.3950
FNR:         0.0740
ROC-AUC:     0.9037

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 No Violence       0.89      0.60      0.72      1000
    Violence       0.70      0.93      0.80      1000

    accuracy                           0.77      2000
   macro avg       0.80      0.77      0.76      2000
weighted avg       0.80      0.77      0.76      2000



2026-03-11 10:36:22,042 - __main__ - INFO - Confusion matrix saved to RLVS_confusion_matrix.png
2026-03-11 10:36:22,337 - __main__ - INFO - ROC curve saved to RLVS_roc_curve.png
2026-03-11 10:36:22,900 - __main__ - INFO - Threshold analysis saved to RLVS_threshold_analysis.png
